In [ ]:
# Required Libraries
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
import json

print("✅ Libraries loaded")

In [ ]:
# Load sample data
titanic = pd.read_csv("../data/titanic.csv")
print(f"📊 Loaded Titanic: {titanic.shape}")

## 1. Report Data Collector

In [ ]:
class ReportDataCollector:
    """
    Collect and structure data for report generation.
    """
    
    @staticmethod
    def collect_overview(df: pd.DataFrame, name: str) -> Dict[str, Any]:
        """
        Collect dataset overview statistics.
        """
        return {
            'name': name,
            'rows': len(df),
            'columns': len(df.columns),
            'numeric_cols': len(df.select_dtypes(include=[np.number]).columns),
            'categorical_cols': len(df.select_dtypes(include=['object']).columns),
            'missing_values': int(df.isna().sum().sum()),
            'missing_pct': round(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2),
            'duplicates': int(df.duplicated().sum()),
            'memory_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 2)
        }
    
    @staticmethod
    def collect_column_stats(df: pd.DataFrame) -> List[Dict[str, Any]]:
        """
        Collect statistics for each column.
        """
        stats = []
        for col in df.columns:
            col_stats = {
                'name': col,
                'dtype': str(df[col].dtype),
                'non_null': int(df[col].notna().sum()),
                'null_count': int(df[col].isna().sum()),
                'null_pct': round(df[col].isna().mean() * 100, 1),
                'unique': int(df[col].nunique())
            }
            
            if df[col].dtype in ['int64', 'float64']:
                col_stats['min'] = round(float(df[col].min()), 2) if pd.notna(df[col].min()) else None
                col_stats['max'] = round(float(df[col].max()), 2) if pd.notna(df[col].max()) else None
                col_stats['mean'] = round(float(df[col].mean()), 2) if pd.notna(df[col].mean()) else None
                col_stats['std'] = round(float(df[col].std()), 2) if pd.notna(df[col].std()) else None
            
            stats.append(col_stats)
        
        return stats
    
    @staticmethod
    def collect_missing_summary(df: pd.DataFrame) -> List[Dict[str, Any]]:
        """
        Collect missing value summary.
        """
        missing = []
        for col in df.columns:
            null_count = df[col].isna().sum()
            if null_count > 0:
                missing.append({
                    'column': col,
                    'missing_count': int(null_count),
                    'missing_pct': round(null_count / len(df) * 100, 1)
                })
        return sorted(missing, key=lambda x: x['missing_pct'], reverse=True)

In [ ]:
# Collect report data
overview = ReportDataCollector.collect_overview(titanic, "Titanic")
column_stats = ReportDataCollector.collect_column_stats(titanic)
missing_summary = ReportDataCollector.collect_missing_summary(titanic)

print("📊 Overview:")
for key, value in overview.items():
    print(f"   {key}: {value}")

## 2. Markdown Report Generator

In [ ]:
class MarkdownReportGenerator:
    """
    Generate formatted Markdown reports.
    """
    
    def __init__(self, title: str):
        self.title = title
        self.content = []
        self.timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    def add_header(self, text: str, level: int = 1):
        """Add a header."""
        self.content.append(f"{'#' * level} {text}\n")
    
    def add_paragraph(self, text: str):
        """Add a paragraph."""
        self.content.append(f"{text}\n")
    
    def add_list(self, items: List[str], ordered: bool = False):
        """Add a list."""
        for i, item in enumerate(items):
            prefix = f"{i+1}." if ordered else "-"
            self.content.append(f"{prefix} {item}")
        self.content.append("")  # Empty line after list
    
    def add_table(self, df: pd.DataFrame):
        """Add a DataFrame as markdown table."""
        self.content.append(df.to_markdown(index=False))
        self.content.append("")  # Empty line after table
    
    def add_key_value_table(self, data: Dict[str, Any], title: str = None):
        """Add a key-value table."""
        if title:
            self.add_header(title, 3)
        
        self.content.append("| Property | Value |")
        self.content.append("|----------|-------|")
        for key, value in data.items():
            self.content.append(f"| {key} | {value} |")
        self.content.append("")  # Empty line
    
    def add_separator(self):
        """Add a horizontal separator."""
        self.content.append("---\n")
    
    def add_code_block(self, code: str, language: str = ""):
        """Add a code block."""
        self.content.append(f"```{language}")
        self.content.append(code)
        self.content.append("```\n")
    
    def generate(self) -> str:
        """Generate the complete markdown report."""
        header = [
            f"# {self.title}",
            f"*Generated: {self.timestamp}*",
            ""
        ]
        return "\n".join(header + self.content)
    
    def save(self, filepath: str):
        """Save report to file."""
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(self.generate())
        print(f"✅ Report saved: {filepath}")

In [ ]:
# Create a markdown report
report = MarkdownReportGenerator("Titanic Dataset Analysis Report")

# Add overview section
report.add_header("Dataset Overview", 2)
report.add_key_value_table(overview)

# Add missing values section
report.add_header("Missing Values", 2)
if missing_summary:
    missing_df = pd.DataFrame(missing_summary)
    report.add_table(missing_df)
else:
    report.add_paragraph("No missing values found.")

# Add column summary
report.add_header("Column Summary", 2)
columns_df = pd.DataFrame(column_stats)[['name', 'dtype', 'non_null', 'null_pct', 'unique']]
report.add_table(columns_df)

# Preview the report
print(report.generate()[:1500])

## 3. HTML Report Generator

In [ ]:
class HTMLReportGenerator:
    """
    Generate styled HTML reports.
    """
    
    CSS_STYLE = """
    <style>
        body { font-family: 'Segoe UI', Arial, sans-serif; margin: 40px; background: #f5f5f5; }
        .container { background: white; padding: 30px; border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }
        h1 { color: #2c3e50; border-bottom: 3px solid #3498db; padding-bottom: 10px; }
        h2 { color: #34495e; margin-top: 30px; }
        h3 { color: #7f8c8d; }
        table { border-collapse: collapse; width: 100%; margin: 20px 0; }
        th, td { border: 1px solid #ddd; padding: 12px; text-align: left; }
        th { background-color: #3498db; color: white; }
        tr:nth-child(even) { background-color: #f9f9f9; }
        tr:hover { background-color: #f1f1f1; }
        .stat-box { display: inline-block; background: #ecf0f1; padding: 15px 25px; margin: 10px; border-radius: 8px; }
        .stat-value { font-size: 24px; font-weight: bold; color: #2c3e50; }
        .stat-label { font-size: 12px; color: #7f8c8d; }
        .timestamp { color: #95a5a6; font-size: 14px; }
        .alert { background: #fff3cd; border: 1px solid #ffc107; padding: 15px; border-radius: 5px; margin: 15px 0; }
        .success { background: #d4edda; border: 1px solid #28a745; padding: 15px; border-radius: 5px; margin: 15px 0; }
    </style>
    """
    
    def __init__(self, title: str):
        self.title = title
        self.sections = []
        self.timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    def add_header(self, text: str, level: int = 2):
        """Add a header."""
        self.sections.append(f"<h{level}>{text}</h{level}>")
    
    def add_paragraph(self, text: str, css_class: str = None):
        """Add a paragraph."""
        class_attr = f' class="{css_class}"' if css_class else ''
        self.sections.append(f"<p{class_attr}>{text}</p>")
    
    def add_stat_boxes(self, stats: Dict[str, Any]):
        """Add statistics as styled boxes."""
        html = '<div class="stats-container">'
        for label, value in stats.items():
            html += f'''
            <div class="stat-box">
                <div class="stat-value">{value}</div>
                <div class="stat-label">{label}</div>
            </div>
            '''
        html += '</div>'
        self.sections.append(html)
    
    def add_table(self, df: pd.DataFrame):
        """Add a DataFrame as HTML table."""
        self.sections.append(df.to_html(index=False, classes='data-table'))
    
    def add_alert(self, message: str, alert_type: str = 'alert'):
        """Add an alert box."""
        self.sections.append(f'<div class="{alert_type}">{message}</div>')
    
    def generate(self) -> str:
        """Generate the complete HTML report."""
        html = f"""
        <!DOCTYPE html>
        <html>
        <head>
            <meta charset="UTF-8">
            <title>{self.title}</title>
            {self.CSS_STYLE}
        </head>
        <body>
            <div class="container">
                <h1>{self.title}</h1>
                <p class="timestamp">Generated: {self.timestamp}</p>
                {''.join(self.sections)}
            </div>
        </body>
        </html>
        """
        return html
    
    def save(self, filepath: str):
        """Save report to file."""
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(self.generate())
        print(f"✅ HTML Report saved: {filepath}")

In [ ]:
# Create HTML report
html_report = HTMLReportGenerator("Titanic Dataset Analysis")

# Add stat boxes
html_report.add_header("Quick Statistics", 2)
html_report.add_stat_boxes({
    'Rows': overview['rows'],
    'Columns': overview['columns'],
    'Missing %': f"{overview['missing_pct']}%",
    'Memory': f"{overview['memory_mb']} MB"
})

# Add missing values alert
if missing_summary:
    html_report.add_header("Missing Values", 2)
    html_report.add_alert(f"Found {len(missing_summary)} columns with missing values")
    html_report.add_table(pd.DataFrame(missing_summary))
else:
    html_report.add_alert("No missing values found!", "success")

# Add column summary
html_report.add_header("Column Summary", 2)
html_report.add_table(pd.DataFrame(column_stats)[['name', 'dtype', 'non_null', 'null_pct', 'unique']])

# Preview (first 1500 chars)
print(html_report.generate()[:1500])

## 4. Complete EDA Report Generator

In [ ]:
class EDAReportGenerator:
    """
    Complete EDA report generator combining all analysis.
    """
    
    def __init__(self, df: pd.DataFrame, name: str):
        self.df = df
        self.name = name
        self.collector = ReportDataCollector()
    
    def generate_markdown(self) -> str:
        """
        Generate a complete Markdown EDA report.
        """
        report = MarkdownReportGenerator(f"EDA Report: {self.name}")
        
        # Overview
        overview = self.collector.collect_overview(self.df, self.name)
        report.add_header("Dataset Overview", 2)
        report.add_key_value_table(overview)
        
        # Column Types
        report.add_header("Column Information", 2)
        report.add_paragraph(f"**Numeric columns ({overview['numeric_cols']}):** {', '.join(self.df.select_dtypes(include=[np.number]).columns.tolist())}")
        report.add_paragraph(f"**Categorical columns ({overview['categorical_cols']}):** {', '.join(self.df.select_dtypes(include=['object']).columns.tolist())}")
        
        # Missing Values
        report.add_header("Missing Values", 2)
        missing = self.collector.collect_missing_summary(self.df)
        if missing:
            report.add_table(pd.DataFrame(missing))
        else:
            report.add_paragraph("✅ No missing values found!")
        
        # Numeric Statistics
        report.add_header("Numeric Column Statistics", 2)
        numeric_stats = self.df.describe().T
        report.add_table(numeric_stats.round(2).reset_index().rename(columns={'index': 'Column'}))
        
        # Categorical Summary
        cat_cols = self.df.select_dtypes(include=['object']).columns
        if len(cat_cols) > 0:
            report.add_header("Categorical Column Summary", 2)
            cat_summary = []
            for col in cat_cols:
                top = self.df[col].mode().iloc[0] if len(self.df[col].mode()) > 0 else 'N/A'
                cat_summary.append({
                    'Column': col,
                    'Unique': self.df[col].nunique(),
                    'Top Value': top,
                    'Missing': self.df[col].isna().sum()
                })
            report.add_table(pd.DataFrame(cat_summary))
        
        # Correlations
        report.add_header("Top Correlations", 2)
        corr = self.df.select_dtypes(include=[np.number]).corr()
        high_corr = []
        for i in range(len(corr.columns)):
            for j in range(i+1, len(corr.columns)):
                if abs(corr.iloc[i, j]) > 0.3:
                    high_corr.append({
                        'Feature 1': corr.columns[i],
                        'Feature 2': corr.columns[j],
                        'Correlation': round(corr.iloc[i, j], 3)
                    })
        if high_corr:
            report.add_table(pd.DataFrame(high_corr))
        else:
            report.add_paragraph("No significant correlations found (threshold: 0.3)")
        
        return report.generate()
    
    def save_markdown(self, output_dir: str = "../reports"):
        """
        Save Markdown report to file.
        """
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"{self.name.lower().replace(' ', '_')}_{timestamp}.md"
        filepath = Path(output_dir) / filename
        
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(self.generate_markdown())
        
        print(f"✅ Report saved: {filepath}")
        return str(filepath)

In [ ]:
# Generate complete EDA report
eda_reporter = EDAReportGenerator(titanic, "Titanic")

# Preview the markdown report
markdown_report = eda_reporter.generate_markdown()
print(markdown_report[:2000])

In [ ]:
# Save the report
# eda_reporter.save_markdown("../reports")

## 5. JSON Report for Programmatic Access

In [ ]:
class JSONReportGenerator:
    """
    Generate JSON reports for programmatic access.
    """
    
    @staticmethod
    def generate(df: pd.DataFrame, name: str) -> Dict[str, Any]:
        """
        Generate a complete JSON report.
        """
        collector = ReportDataCollector()
        
        report = {
            'metadata': {
                'name': name,
                'generated_at': datetime.now().isoformat(),
                'version': '1.0'
            },
            'overview': collector.collect_overview(df, name),
            'columns': collector.collect_column_stats(df),
            'missing_values': collector.collect_missing_summary(df),
            'numeric_summary': df.describe().to_dict(),
            'dtypes': df.dtypes.astype(str).to_dict()
        }
        
        return report
    
    @staticmethod
    def save(report: Dict, filepath: str):
        """Save JSON report to file."""
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(report, f, indent=2, default=str)
        print(f"✅ JSON Report saved: {filepath}")

In [ ]:
# Generate JSON report
json_report = JSONReportGenerator.generate(titanic, "Titanic")

# Preview structure
print("📊 JSON Report Structure:")
print(f"   Keys: {list(json_report.keys())}")
print(f"\n📋 Metadata:")
for key, value in json_report['metadata'].items():
    print(f"   {key}: {value}")

In [ ]:
# Access specific data programmatically
print("\n🔢 Overview Stats:")
for key, value in json_report['overview'].items():
    print(f"   {key}: {value}")

## ✅ Summary

This reporting module provides:

**1. ReportDataCollector**
- Collect structured data for reports
- Overview statistics
- Column-level analysis
- Missing value summaries

**2. MarkdownReportGenerator**
- Headers, paragraphs, lists
- Tables from DataFrames
- Key-value tables
- Code blocks

**3. HTMLReportGenerator**
- Styled HTML output
- Stat boxes
- Alert messages
- Professional formatting

**4. EDAReportGenerator**
- Complete EDA reports
- All sections combined
- File export

**5. JSONReportGenerator**
- Programmatic access
- Structured data export
- API-friendly format